
# Assignment 3 for Course 1MS041
Make sure you pass the `# ... Test` cells and
 submit your solution notebook in the corresponding assignment on the course website. You can submit multiple times before the deadline and your highest score will be used.

---
## Assignment 3, PROBLEM 1
Maximum Points = 8


Download the updated data folder from the course github website or just download directly the file [https://github.com/datascience-intro/1MS041-2025/blob/main/notebooks/data/smhi.csv](https://github.com/datascience-intro/1MS041-2025/blob/main/notebooks/data/smhi.csv) from the github website and put it inside your data folder, i.e. you want the path `data/smhi.csv`. The data was aquired from SMHI (Swedish Meteorological and Hydrological Institute) and constitutes per hour measurements of wind in the Uppsala Aut station. The data consists of windspeed and direction. Your goal is to load the data and work with it a bit. The code you produce should load the file as it is, please do not alter the file as the autograder will only have access to the original file.

The file information is in Swedish so you need to use some translation service, for instance `Google translate` or ChatGPT.

1. [2p] Load the file, for instance using the `csv` package. Put the wind-direction as a numpy array and the wind-speed as another numpy array.
2. [2p] Use the wind-direction (see [Wikipedia](https://en.wikipedia.org/wiki/Wind_direction)) which is an angle in degrees and convert it into a point on the unit circle **which is the direction the wind is blowing to** (compare to definition of radians [Wikipedia](https://en.wikipedia.org/wiki/Radian)). Store the `x_coordinate` as one array and the `y_coordinate` as another. From these coordinates, construct the wind-velocity vector.
3. [2p] Calculate the average wind velocity and convert it back to direction and compare it to just taking average of the wind direction as given in the data-file.
4. [2p] The wind velocity is a $2$-dimensional random variable, calculate the empirical covariance matrix which should be a numpy array of shape (2,2).

For you to wonder about, is it more likely for you to have headwind or not when going to the university in the morning.

In [ ]:
import numpy as np
import re

# Bangla: SMHI file ta semicolon (;) separated, ebong shurute Swedish metadata ache.
# Tai amra puro file text hisebe read kore sudhu data rows gulo regex diye ber korbo.
wind_directions = []
wind_speeds = []

with open("data/smhi.csv", "r", encoding="utf-8-sig") as f:
    text = f.read()

# Bangla: prottek real data row er format roughly:
# YYYY-MM-DD;HH:MM:SS;Vindriktning;Quality;Vindhastighet;Quality
pattern = r"(\d{4}-\d{2}-\d{2});(\d{2}:\d{2}:\d{2});([^;]+);[^;]*;([^;]+);"

for match in re.finditer(pattern, text):
    direction_text = match.group(3).replace(",", ".")
    speed_text = match.group(4).replace(",", ".")
    try:
        direction = float(direction_text)
        speed = float(speed_text)
        # Bangla: missing/invalid value thakle skip korbo.
        if np.isfinite(direction) and np.isfinite(speed):
            wind_directions.append(direction)
            wind_speeds.append(speed)
    except ValueError:
        pass

# Bangla: final answer numpy array hote hobe.
problem1_wind_direction = np.array(wind_directions, dtype=float)
problem1_wind_speed = np.array(wind_speeds, dtype=float)


In [ ]:
# The wind direction is given as a compass direction in degrees (0-360)
# convert it to x and y coordinates using the standard mathematical convention

# Bangla: Wind direction usually bole wind kon dik THEKE ashche.
# Assignment e bola ache direction wind is blowing TO lagbe, tai 180 degree add kori.
wind_direction_to = (problem1_wind_direction + 180) % 360

# Bangla: Compass angle: 0=N, 90=E, clockwise.
# Math angle: 0=positive x/east, 90=positive y/north, counter-clockwise.
# Tai compass theke math angle e convert: math_angle = 90 - compass_angle.
math_angle_degrees = (90 - wind_direction_to) % 360
math_angle_radians = np.deg2rad(math_angle_degrees)

# Bangla: Unit circle e x = cos(angle), y = sin(angle).
problem1_wind_direction_x_coordinate = np.cos(math_angle_radians)
problem1_wind_direction_y_coordinate = np.sin(math_angle_radians)

# Bangla: Velocity vector = speed * direction vector.
problem1_wind_velocity_x_coordinate = problem1_wind_speed * problem1_wind_direction_x_coordinate
problem1_wind_velocity_y_coordinate = problem1_wind_speed * problem1_wind_direction_y_coordinate


In [ ]:
# Put the average wind velocity x and y coordinates here in these variables

# Bangla: Average velocity vector pete x-component ebong y-component er mean nei.
problem1_average_wind_velocity_x_coordinate = np.mean(problem1_wind_velocity_x_coordinate)
problem1_average_wind_velocity_y_coordinate = np.mean(problem1_wind_velocity_y_coordinate)

# First calculate the angle of the average wind velocity vector in degrees
# Bangla: atan2(y,x) vector er mathematical angle dey. Degree te convert kore 0-360 range e rakhi.
problem1_average_wind_velocity_angle_degrees = (
    np.rad2deg(np.arctan2(problem1_average_wind_velocity_y_coordinate,
                         problem1_average_wind_velocity_x_coordinate)) % 360
)

# Then calculate the average angle of the wind direction in degrees (using the wind direction in the data)
# Bangla: Eta simple arithmetic average. Circular data te eta generally correct average na.
problem1_average_wind_direction_angle_degrees = np.mean(problem1_wind_direction)

# Finally, are they the same? Answer as a boolean value (True or False)
# Bangla: Vector average angle and raw direction average generally same hoy na.
problem1_same_angle = bool(np.isclose(problem1_average_wind_velocity_angle_degrees,
                                      problem1_average_wind_direction_angle_degrees))


In [ ]:
# Bangla: Wind velocity 2D random variable: [Vx, Vy].
# np.cov(..., rowvar=False) dile column-wise covariance matrix pabo, shape (2,2).
problem1_wind_velocity_covariance_matrix = np.cov(
    np.column_stack((problem1_wind_velocity_x_coordinate,
                     problem1_wind_velocity_y_coordinate)),
    rowvar=False
)


---
## Assignment 3, PROBLEM 2
Maximum Points = 8


For this problem you will need the [pandas](https://pandas.pydata.org/) package and the [sklearn](https://scikit-learn.org/stable/) package. Inside the `data` folder from the course website you will find a file called `indoor_train.csv`, this file includes a bunch of positions in (X,Y,Z) and also a location number. The idea is to assign a room number (Location) to the coordinates (X,Y,Z).

1. [2p] Take the data in the file `indoor_train.csv` and load it using pandas into a dataframe `df_train`
2. [3p] From this dataframe `df_train`, create two numpy arrays, one `Xtrain` and `Ytrain`, they should have sizes `(1154,3)` and `(1154,)` respectively. Their `dtype` should be `float64` and `int64` respectively.
3. [3p] Train a Support Vector Classifier, `sklearn.svc.SVC`, on `Xtrain, Ytrain` with `kernel='linear'` and name the trained model `svc_train`.

To mimic how [kaggle](https://www.kaggle.com/) works, the Autograder has access to a hidden test-set and will test your fitted model.

In [ ]:
import pandas as pd

# Bangla: indoor_train.csv file ta pandas dataframe hisebe load korchi.
df_train = pd.read_csv("data/indoor_train.csv")


In [ ]:
import numpy as np

# Bangla: Feature columns hobe position coordinates X, Y, Z.
# Jodi column name exact X,Y,Z hoy, directly nibo. Otherwise Location bad diye first 3 numeric column nibo.
if all(col in df_train.columns for col in ["X", "Y", "Z"]):
    feature_columns = ["X", "Y", "Z"]
else:
    feature_columns = [col for col in df_train.columns if col.lower() != "location"][:3]

# Bangla: Target/label column holo Location.
if "Location" in df_train.columns:
    target_column = "Location"
else:
    target_column = [col for col in df_train.columns if col not in feature_columns][-1]

# Bangla: Autograder je dtype chay: Xtrain float64, Ytrain int64.
Xtrain = df_train[feature_columns].to_numpy(dtype=np.float64)
Ytrain = df_train[target_column].to_numpy(dtype=np.int64)


In [ ]:
from sklearn.svm import SVC

# Bangla: Linear kernel diye Support Vector Classifier train korchi.
svc_train = SVC(kernel="linear")
svc_train.fit(Xtrain, Ytrain)


---
## Assignment 3, PROBLEM 3
Maximum Points = 8


Let us build a proportional model ($\mathbb{P}(Y=1 \mid X) = G(\beta_0+\beta \cdot X)$ where $G$ is the logistic function) for the spam vs not spam data. Here we assume that the features are presence vs not presence of a word, let $X_1,X_2,X_3$ denote the presence (1) or absence (0) of the words $("free", "prize", "win")$.

1. [2p] Load the file `data/spam.csv` and create two numpy arrays, `problem3_X` which has shape **(n_texts,3)** where each feature in `problem3_X` corresponds to $X_1,X_2,X_3$ from above, `problem3_Y` which has shape **(n_texts,)** and consists of a $1$ if the email is spam and $0$ if it is not. Split this data into a train-calibration-test sets where we have the split $40\%$, $20\%$, $40\%$, put this data in the designated variables in the code cell.

2. [2p] Follow the calculation from the lecture notes where we derive the logistic regression and implement the final loss function inside the class `ProportionalSpam`. You can use the `Test` cell to check that it gives the correct value for a test-point.

3. [2p] Train the model `problem3_ps` on the training data. The goal is to calibrate the probabilities output from the model. Start by creating a new variable `problem3_X_pred` (shape `(n_samples,1)`) which consists of the predictions of `problem3_ps` on the calibration dataset. Then train a calibration model using `sklearn.tree.DecisionTreeRegressor`, store this trained model in `problem3_calibrator`. Recall that calibration error is the following for a fixed function $f$
$$
    \sqrt{\mathbb{E}[|\mathbb{E}[Y \mid f(X)] - f(X)|^2]}.
$$

4. [2p] Use the trained model `problem3_ps` and the calibrator `problem3_calibrator` to make final predictions on the testing data, store the prediction in `problem3_final_predictions`. 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# spam.csv file read korchi
df = pd.read_csv("data/spam.csv")

# Je 3 ta word check korte bolse
words = ["free", "prize", "win"]

# X banabo: word thakle 1, na thakle 0
problem3_X = np.array([
    [1 if word in text.lower() else 0 for word in words]
    for text in df["v2"]
])

# Y banabo: spam hole 1, ham hole 0
problem3_Y = np.array([
    1 if label == "spam" else 0
    for label in df["v1"]
])

# First split: 40% train, 60% remaining
problem3_X_train, X_temp, problem3_Y_train, Y_temp = train_test_split(
    problem3_X,
    problem3_Y,
    train_size=0.4,
    random_state=0
)

# Remaining 60% theke calibration 20%, test 40%
# Since temp = 60%, calibration should be 1/3 of temp
problem3_X_calib, problem3_X_test, problem3_Y_calib, problem3_Y_test = train_test_split(
    X_temp,
    Y_temp,
    train_size=1/3,
    random_state=0
)

print(
    problem3_X_train.shape,
    problem3_X_calib.shape,
    problem3_X_test.shape,
    problem3_Y_train.shape,
    problem3_Y_calib.shape,
    problem3_Y_test.shape
)

In [ ]:
class ProportionalSpam(object):
    def __init__(self):
        self.coeffs = None
        self.result = None
    
    # Ei function loss calculate kore
    def loss(self, X, Y, coeffs):
        import numpy as np
        
        # beta0 holo intercept
        beta0 = coeffs[0]
        
        # baki gula beta coefficient
        beta = coeffs[1:]
        
        # linear part: beta0 + beta.X
        z = beta0 + np.dot(X, beta)
        
        # logistic function G(z)
        p = np.exp(z) / (1 + np.exp(z))
        
        # numerical problem avoid korar jonno p ke tiny range er moddhe rakhi
        p = np.clip(p, 1e-10, 1 - 1e-10)
        
        # logistic regression negative log likelihood
        loss_value = -np.mean(Y * np.log(p) + (1 - Y) * np.log(1 - p))
        
        return loss_value

    def fit(self, X, Y):
        import numpy as np
        from scipy import optimize

        # loss minimize kore best coefficients ber korbo
        opt_loss = lambda coeffs: self.loss(X, Y, coeffs)
        
        # initial coefficients sob 0
        initial_arguments = np.zeros(shape=X.shape[1] + 1)
        
        # optimization
        self.result = optimize.minimize(opt_loss, initial_arguments, method='cg')
        
        # final trained coefficients save korchi
        self.coeffs = self.result.x
    
    def predict(self, X):
        import numpy as np
        
        # trained model diye probability predict korbo
        if self.coeffs is not None:
            G = lambda x: np.exp(x) / (1 + np.exp(x))
            
            # output 0.1, 0.2, 0.3 er moto rounded probability hobe
            return np.round(
                10 * G(np.dot(X, self.coeffs[1:]) + self.coeffs[0])
            ) / 10

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# model object create korchi
problem3_ps = ProportionalSpam()

# training data diye model train korchi
problem3_ps.fit(problem3_X_train, problem3_Y_train)

# calibration data te model er probability prediction nicchi
# reshape(-1, 1) dorkar karon DecisionTreeRegressor 2D input chay
problem3_X_pred = problem3_ps.predict(problem3_X_calib).reshape(-1, 1)

# calibrator model create korchi
problem3_calibrator = DecisionTreeRegressor(random_state=0)

# calibrator train korchi
# input: model er predicted probability
# output: actual spam/ham label
problem3_calibrator.fit(problem3_X_pred, problem3_Y_calib)

In [ ]:
# test data te first logistic model diye probability predict korchi
test_pred = problem3_ps.predict(problem3_X_test).reshape(-1, 1)

# calibrator diye final calibrated prediction nicchi
problem3_final_predictions = problem3_calibrator.predict(test_pred)

---
#### Local Test for Assignment 3, PROBLEM 3
Evaluate cell below to make sure your answer is valid.                             You **should not** modify anything in the cell below when evaluating it to do a local test of                             your solution.
You may need to include and evaluate code snippets from lecture notebooks in cells above to make the local test work correctly sometimes (see error messages for clues). This is meant to help you become efficient at recalling materials covered in lectures that relate to this problem. Such local tests will generally not be available in the exam.

In [ ]:
try:
    import numpy as np
    test_instance = ProportionalSpam()
    test_loss = test_instance.loss(np.array([[1,0,1],[0,1,1]]),np.array([1,0]),np.array([1.2,0.4,0.3,0.9]))
    assert (np.abs(test_loss-1.2828629432232497) < 1e-6)
    print("Your loss was correct for a test point")
except:
    print("Your loss was not correct on a test point")